# Apache Beam Pipeline Examples

This notebook demonstrates various Apache Beam concepts, including:

*   **Custom Coders**: Defining custom serialization/deserialization logic for Python objects (`Employee` class).
*   **Type Hints**: Using `with_input_types` and `with_output_types` for type checking in Beam PCollections and DoFns.

In [ ]:
!pip install --quiet apache-beam[gcp]

In [10]:
import apache_beam as beam
import typing
p1=beam.Pipeline()

class Employee(object):
  def __init__(self,id,name):
    self.id=id
    self.name=name

class EmployeeCoder(beam.coders.Coder):
  def encode(self,employee):
    return ('%s:%s' %(employee.id,employee.name)).encode('utf8')

  def decode(self,s):
    return Employee(*s.decode('utf8').split(':'))

  def is_deterministic(self):
    return True

beam.coders.registry.register_coder(Employee,EmployeeCoder)

def splitData(element):
  id,name,salary=element.split(',')
  return Employee(id,name),int(salary)

custom_coders_pcoll = (
    p1
    |"Read input file">>beam.io.ReadFromText('/content/data.txt')
    |"split data">>beam.Map(splitData)
    |"combine data">>beam.CombinePerKey(sum).with_input_types(typing.Tuple[Employee,int])
    |"write output">>beam.io.WriteToText('/content/output.txt')

)
p1.run()

!{('head -n 5 /content/output*')}

(<__main__.Employee object at 0x7e664a771700>, 60000)
(<__main__.Employee object at 0x7e664a723bc0>, 120000)


In [ ]:
#typehints
import apache_beam as beam
p=beam.Pipeline()

#with_input_types

typehints_pcoll=(
    p
    |"Read input file">>beam.Create(['one','two','three'])
    |"Filter even elements">>beam.Filter(lambda element:element%2==0).with_input_types(int)
)
p.run()



In [ ]:
import apache_beam as beam
p=beam.Pipeline()

@beam.typehints.with_input_types(int)
@beam.typehints.with_output_types(int)
class FilterDoFn(beam.DoFn):
  def process(self,element):
    if element%2==0:
      yield element

type_hints_pcoll=(
    p|
    beam.Create(['1','2','3','4','5'])
    |beam.ParDo(FilterDoFn())
)